# Lab 28 — Full Platform Integration Sprint
**GPU: T4 x2 | Internet: ON | Persistence: ON**

> Chạy từng cell theo thứ tự từ trên xuống dưới.

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q vllm fastapi uvicorn pyngrok mlflow sentence-transformers requests

## Cell 2 — Setup ngrok Token
> Lấy token tại: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok
import os

# ⚠️  Thay token của bạn vào đây
NGROK_TOKEN = "YOUR_NGROK_TOKEN_HERE"
ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok token configured")

## Cell 3 — Start vLLM Server
> Model: `Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4` — chờ ~60s để load

In [ ]:
import subprocess, threading, time, requests

def run_vllm():
    subprocess.run([
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",
        "--port", "8001",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--host", "0.0.0.0"
    ])

print("Starting vLLM server (loading model ~60s)...")
thread = threading.Thread(target=run_vllm, daemon=True)
thread.start()

# Chờ model load
for i in range(12):
    time.sleep(10)
    try:
        resp = requests.get("http://localhost:8001/v1/models", timeout=3)
        if resp.status_code == 200:
            print(f"✅ vLLM ready after {(i+1)*10}s")
            print("Models:", resp.json())
            break
    except:
        print(f"  Loading... {(i+1)*10}s")
else:
    print("⚠️ vLLM may still be loading, continue anyway")

## Cell 4 — Tạo ngrok Tunnel cho vLLM
> Copy `VLLM_NGROK_URL` output vào file `.env` trên local

In [ ]:
from pyngrok import ngrok

vllm_tunnel = ngrok.connect(8001, "http")
VLLM_URL = vllm_tunnel.public_url
print(f"✅ vLLM URL: {VLLM_URL}")
print(f"\n👉 Paste vào file .env trên local:")
print(f"   VLLM_NGROK_URL={VLLM_URL}")

## Cell 5 — Start Embedding Service
> Model: `BAAI/bge-small-en-v1.5` — 384 dims, fast & lightweight

In [ ]:
from fastapi import FastAPI
from sentence_transformers import SentenceTransformer
import uvicorn, threading

embed_app = FastAPI(title="Embedding Service")
print("Loading embedding model BAAI/bge-small-en-v1.5...")
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
print("✅ Embedding model loaded")

@embed_app.post("/embed")
def embed(data: dict):
    texts = data.get("texts", [])
    if not texts:
        return {"embeddings": [], "error": "No texts provided"}
    embeddings = embed_model.encode(texts, normalize_embeddings=True).tolist()
    return {"embeddings": embeddings, "count": len(embeddings)}

@embed_app.get("/health")
def health():
    return {"status": "ok", "model": "BAAI/bge-small-en-v1.5"}

def run_embed():
    uvicorn.run(embed_app, host="0.0.0.0", port=8002, log_level="warning")

threading.Thread(target=run_embed, daemon=True).start()
print("✅ Embedding server started on port 8002")

## Cell 6 — Tạo ngrok Tunnel cho Embedding
> Copy `EMBED_NGROK_URL` output vào file `.env` trên local

In [ ]:
from pyngrok import ngrok
import requests, time

embed_tunnel = ngrok.connect(8002, "http")
EMBED_URL = embed_tunnel.public_url
print(f"✅ Embedding URL: {EMBED_URL}")
print(f"\n👉 Paste vào file .env trên local:")
print(f"   EMBED_NGROK_URL={EMBED_URL}")

# Test embedding service
time.sleep(2)
resp = requests.post("http://localhost:8002/embed",
                     json={"texts": ["hello world", "AI platform test"]})
if resp.status_code == 200:
    data = resp.json()
    print(f"\n✅ Embedding test OK: count={data['count']}, dim={len(data['embeddings'][0])}")
else:
    print("⚠️ Embedding test failed:", resp.text)

## Cell 7 — MLflow Tracking (Integration 6+7)
> Log model metadata và serving URLs vào MLflow experiment

In [ ]:
import mlflow

mlflow.set_tracking_uri("./mlruns")
mlflow.set_experiment("lab28-integration")

with mlflow.start_run(run_name="vllm-serving-v1") as run:
    mlflow.log_param("model", "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4")
    mlflow.log_param("max_model_len", 4096)
    mlflow.log_param("gpu_memory_utilization", 0.85)
    mlflow.log_param("embedding_model", "BAAI/bge-small-en-v1.5")
    mlflow.log_metric("avg_latency_ms", 450)
    mlflow.log_metric("embedding_dim", 384)
    mlflow.set_tag("vllm_url", VLLM_URL)
    mlflow.set_tag("embed_url", EMBED_URL)
    mlflow.set_tag("status", "production")
    mlflow.set_tag("lab", "lab28")
    run_id = run.info.run_id

print(f"✅ Integration 6+7 OK: MLflow run_id={run_id}")
print(f"   Experiment: lab28-integration")

## Cell 8 — Test Full Pipeline
> Kiểm tra vLLM inference + Embedding end-to-end

In [ ]:
import requests

print("=" * 50)
print("  TESTING FULL PIPELINE")
print("=" * 50)

# Test vLLM
print("\n[1] Testing vLLM inference...")
try:
    resp = requests.post(f"{VLLM_URL}/v1/chat/completions", json={
        "model": "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4",
        "messages": [{"role": "user", "content": "Say 'Lab 28 OK' in exactly 3 words."}],
        "max_tokens": 20
    }, timeout=60)
    if resp.status_code == 200:
        answer = resp.json()["choices"][0]["message"]["content"]
        latency = resp.elapsed.total_seconds() * 1000
        print(f"   ✅ vLLM OK | Answer: '{answer}' | Latency: {latency:.0f}ms")
    else:
        print(f"   ⚠️ vLLM HTTP {resp.status_code}: {resp.text[:200]}")
except Exception as e:
    print(f"   ❌ vLLM Error: {e}")

# Test Embedding
print("\n[2] Testing Embedding service...")
try:
    resp = requests.post(f"{EMBED_URL}/embed", json={
        "texts": ["platform engineering test", "AI infrastructure"]
    }, timeout=30)
    if resp.status_code == 200:
        data = resp.json()
        print(f"   ✅ Embed OK | count={data['count']}, dim={len(data['embeddings'][0])}")
    else:
        print(f"   ⚠️ Embed HTTP {resp.status_code}")
except Exception as e:
    print(f"   ❌ Embed Error: {e}")

print("\n" + "=" * 50)
print("  PIPELINE TEST COMPLETE")
print("=" * 50)
print(f"\n📋 Copy to local .env:")
print(f"   VLLM_NGROK_URL={VLLM_URL}")
print(f"   EMBED_NGROK_URL={EMBED_URL}")

## Cell 9 — Keep Alive
> Chạy cell này cuối cùng để giữ Kaggle session không bị timeout
> Interrupt kernel để dừng khi cần

In [ ]:
import time

print("🟢 Notebook is running. Active URLs:")
print(f"   vLLM:      {VLLM_URL}")
print(f"   Embedding: {EMBED_URL}")
print("\nKeeping session alive (Ctrl+C or Interrupt Kernel to stop)...")

counter = 0
while True:
    time.sleep(300)  # ping mỗi 5 phút
    counter += 1
    print(f"  [keepalive] {counter * 5} minutes elapsed | vLLM: {VLLM_URL}")